# 第 8 集 — 一个函数就是一个 synth:参数与 preset

⏱ 约 25 分钟 · 难度 ★★☆

**这一集控制的音乐元素:全部** — pitch、timbre、brightness、articulation、movement,第一次同时出现。

**Max / RNBO 里的对应:** 一个完整的 RNBO patch,顶上一排 `param` object,里面是 osc → filter → VCA,外加两个 envelope。

**你会听到:** 三个 preset —
1. `pluck`:短促、开头亮的拨弦音
2. `pad`:慢慢淡入、逐渐变亮的长音
3. `wow`:低音,filter 和 wavetable 一起慢慢张开再合上,"wow——"

然后用这个 synth 弹一段旋律。

**学完你能做到:**
- 用 `dict` 存一组参数
- 读懂 `synth()` 的每一行,知道它对应哪一集
- 改参数、做自己的 preset
- 用一个 preset 弹出一串音

这一集没有新的 DSP,只有 "接线" 和一个新的 Python 工具:`dict`。

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # 时间不能是 0,否则下面会除以 0
    # 按住期间:(0,0) → (a,1) → (a+d,s),之后停在 s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # 松开那一刻 envelope 有多高(可能还没走完 attack)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # 从那个高度用 r 秒直线降到 0;np.clip 让它降到 0 后不再变负
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # 松开前用 held,松开后用 released

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff 给一个数或一个 array 都行:统一变成和声音一样长的 array
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- 把 cutoff / res 换算成 filter 内部用的系数(每个 sample 一组)---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping:res 越大 k 越小,共鸣越强
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() 把 array 变成普通 list:在循环里逐个取数时快很多
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # 两个状态变量 = gen~ 里的两个 history

    for n in range(n_samples):                 # 一个 sample 一个 sample 按顺序算
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 就是 lowpass 的输出
        ic1 = 2 * v1 - ic1                     # 更新状态,留给下一个 sample 用
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = 这一个 sample 里 phase 往前走多少
    # cumsum 把每一步累加起来;% 1 只留小数部分
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

frames = make_frames()

## 1. Signal flow

```
note ──► mtof ──► phasor ──► wavetable osc ──► lowpass ──► × ──► out
                                 ▲                ▲         ▲
                              wt_pos           cutoff    amp env
                                 ▲                ▲
                                 └──── mod env ───┘
```

两个 envelope:

- **amp env**:控制音量(第 5 集)
- **mod env**:同时去推 wavetable position 和 cutoff(第 7 集),各自有一个 amount

## 2. `dict`:有名字的参数

到目前为止,参数都是一个个单独的变量(`cutoff = 300`、`res = 0.5` ……)。参数一多就很乱。

RNBO 里每个 `param` 有 **名字** 和 **值**。Python 里对应的是 `dict`:一张 **名字 → 值** 的表。Max 里也有同名的 `dict` object,是同一个东西。

- 建立:`{"名字": 值, "名字": 值}`
- 取值:`p["名字"]`
- 改值:`p["名字"] = 新值`

In [ ]:
p = {"note": 45, "cutoff": 300, "res": 0.2}

print("the whole dict :", p)
print("p['cutoff']    :", p["cutoff"])

p["cutoff"] = 1200                             # 拧了一下 cutoff 旋钮
print("after change   :", p)

for name in p:                                 # 依次取出每个名字
    print(name, "=", p[name])

还有一种写法 `dict(note=45, cutoff=300)`,结果完全一样,只是名字不用加引号,打起来快。下面的 preset 用这种写法。

## 3. `mtof`

Max 的 `mtof`:MIDI 音符号 → Hz。

- MIDI 69 = A4 = 440 Hz
- 每高 12 个半音,频率翻一倍(第 7 集的 `2 ** 八度数`)

In [ ]:
def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # (m - 69) / 12 = 离 A4 几个八度

for m in [69, 57, 45, 33, 60]:
    print("MIDI", m, "->", round(mtof(m), 2), "Hz")

## 4. 一步一步搭起来

不直接给出最终版本,而是分三步搭,每一步都听一下。

### 第一步:oscillator + amp envelope

只用到 6 个参数:`note`、`wt_pos`、`amp_a`、`amp_d`、`amp_s`、`amp_r`。

函数的第二、三个 inlet 是 `dur`(整段多长)和 `gate`(按住多久),都有默认值。

**应该听到:** 一个 A2 的音,没有 filter,音色全程不变,只有音量在动。

In [ ]:
def synth_v1(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    phase = phasor(mtof(p["note"]), n)                     # note → Hz → phase
    x = osc_wavetable(frames, phase, p["wt_pos"])
    return x * amp_env

test = dict(note=45, wt_pos=0.6,
            amp_a=0.01, amp_d=0.3, amp_s=0.5, amp_r=0.3)

play(synth_v1(test))

### 第二步:加上 filter

多两个参数:`cutoff`、`res`。只多了一行。

**应该听到:** 同一个音,变闷了(cutoff 600 Hz),带一点共鸣。

In [ ]:
def synth_v2(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"])
    x = lowpass(x, p["cutoff"], p["res"])                  # 新加的一行
    return x * amp_env

test = dict(test, cutoff=600, res=0.4)                     # 复制 test,并加上两个新参数
play(synth_v2(test))

`dict(test, cutoff=600, res=0.4)`:复制一份 `test`,并加上(或改掉)后面写的那几项。原来的 `test` 不会被动到 — 不过这里我们又把结果存回了 `test` 这个名字。

### 第三步:加上 mod envelope

再多 6 个参数:

- `mod_a`、`mod_d`、`mod_s`、`mod_r`:第二条 envelope
- `wt_env`:它推 wavetable position 推多少(0–1)
- `cut_env`:它推 cutoff 推多少(八度)

`wt_pos` 和 `cutoff` 不再是固定的数,而是 **旋钮的值 + envelope 推上去的量**。这正是第 7 集做的事。

这就是最终版本,一共 14 个参数。

In [ ]:
def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # 两个 envelope
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator:wt_pos 旋钮 + mod env 推上去的量
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter:cutoff 旋钮,被 mod env 往上推 cut_env 个八度
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

逐行对照:

| 行 | 做什么 | 哪一集 |
|---|---|---|
| `t = ...` | 时间表 | 1 |
| `amp_env`、`mod_env` | 两个 `adsr~` | 5 |
| `phase = phasor(mtof(...))` | `mtof` → `phasor~` | 7 |
| `osc_wavetable(...)` | position = 旋钮 + amount × mod env | 4、7 |
| `lowpass(...)` | cutoff = 旋钮,被 mod env 往上推 `cut_env` 个八度 | 6、7 |
| `x * amp_env` | VCA(`*~`) | 5 |

### 14 个参数

| 名字 | 意思 | 建议范围 |
|---|---|---|
| `note` | MIDI 音高 | 28–60 |
| `wt_pos` | wavetable position | 0–1 |
| `wt_env` | mod env → wt_pos 的 amount | 0–1 |
| `cutoff` | filter cutoff (Hz) | 80–8000 |
| `res` | resonance | 0–0.9 |
| `cut_env` | mod env → cutoff 的 amount(八度) | 0–5 |
| `amp_a` `amp_d` `amp_s` `amp_r` | amp envelope | 时间:秒;`amp_s`:0–1 |
| `mod_a` `mod_d` `mod_s` `mod_r` | mod envelope | 同上 |

## 5. Preset

一个 preset 就是一个填好了的 `dict`。`presets` 是一个 "装着 dict 的 dict":名字 → preset。

In [ ]:
presets = {
    "pluck": dict(note=45, wt_pos=0.6, wt_env=0.0, cutoff=300, res=0.2, cut_env=4.0,
                  amp_a=0.005, amp_d=0.30, amp_s=0.0, amp_r=0.10,
                  mod_a=0.001, mod_d=0.15, mod_s=0.0, mod_r=0.10),
    "pad":   dict(note=57, wt_pos=0.2, wt_env=0.5, cutoff=800, res=0.1, cut_env=2.0,
                  amp_a=0.60, amp_d=0.30, amp_s=0.8, amp_r=0.70,
                  mod_a=1.00, mod_d=0.50, mod_s=0.6, mod_r=0.50),
    "wow":   dict(note=33, wt_pos=0.3, wt_env=0.7, cutoff=150, res=0.6, cut_env=4.5,
                  amp_a=0.01, amp_d=0.10, amp_s=0.9, amp_r=0.10,
                  mod_a=0.35, mod_d=0.40, mod_s=0.2, mod_r=0.20),
}

### 一个 "看 preset" 的小工具

听之前先看:`show(p)` 画出这个 preset 的两条 envelope,以及 cutoff 随时间的实际曲线。这相当于 Serum 界面上那几个 envelope 窗口。

In [ ]:
def show(p, dur=2.0, gate=1.2):
    t = np.arange(int(sr * dur)) / sr
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)
    fig, axes = plt.subplots(1, 2, figsize=(11, 2.5))
    axes[0].plot(t, amp_env, label="amp env")
    axes[0].plot(t, mod_env, label="mod env")
    axes[0].axvline(gate, linestyle="--", color="gray")
    axes[0].legend()
    axes[0].set_xlabel("time (s)")
    axes[1].plot(t, p["cutoff"] * 2 ** (p["cut_env"] * mod_env))      # 和 synth() 里算 cutoff 的式子一样
    axes[1].set_ylabel("cutoff (Hz)")
    axes[1].set_xlabel("time (s)")
    plt.show()

### pluck

**应该听到:** 短促的拨弦音,开头有一个亮的 "嗒",随后迅速变闷消失。

**看图:** mod env 比 amp env 更短,所以先变闷、再变小声。cutoff 从 4800 Hz 在 0.15 秒内掉到 300 Hz。

In [ ]:
show(presets["pluck"])
play(synth(presets["pluck"]))

### pad

**应该听到:** 一个中音区的长音,慢慢淡入,同时逐渐变亮,松开后缓缓消失。

**看图:** 两条 envelope 的 attack 都很长;mod env 用 1 秒升到顶,所以亮度的变化比音量更慢。

In [ ]:
show(presets["pad"])
play(synth(presets["pad"]))

### wow

**应该听到:** 一个很低的 bass,音色像嘴巴慢慢张开再合上:"wow——"。

**看图:** amp env 几乎立刻到顶并保持;mod env 用 0.35 秒升上去再落下来 — "wow" 的形状就是 mod env 的形状。

In [ ]:
show(presets["wow"])
play(synth(presets["wow"]))

## 6. 拧旋钮

Python 里 "拧旋钮" 不是实时的:**改数字 → 重新算一遍 → 再听**。

好处是可以让循环帮你拧:一次听同一个旋钮的好几个位置。

### `cutoff`

**应该听到:** 同一个 `pluck`,四次,一次比一次亮。

In [ ]:
for value in [100, 300, 900, 2700]:
    print("cutoff =", value)
    display(play(synth(dict(presets["pluck"], cutoff=value))))      # 复制 pluck,只换掉 cutoff

### `cut_env`

**应该听到:** `wow` 四次。amount 为 0 时完全没有 "wow"(filter 不动);越大张得越开。

In [ ]:
for value in [0, 1.5, 3, 4.5]:
    print("cut_env =", value)
    display(play(synth(dict(presets["wow"], cut_env=value))))

### `mod_a`

**应该听到:** `wow` 四次。"wow" 张开的速度从很快("wa")到很慢("wooooow")。

In [ ]:
for value in [0.02, 0.15, 0.35, 0.8]:
    print("mod_a =", value)
    display(play(synth(dict(presets["wow"], mod_a=value))))

## 7. 用 preset 弹一段音

和第 5 集的 bassline 一样:循环里每次换一个 `note`,渲染一个短音,最后接起来。

**应该听到:** 用 `pluck` 音色弹的一段 8 个音的琶音,A 小调。

In [ ]:
def play_notes(p, notes, dur=0.3):
    out = []
    for m in notes:
        one = synth(dict(p, note=m), dur=dur, gate=dur * 0.6)      # 换音高,其余参数不变
        out.append(one)
    return np.concatenate(out)

arp = [45, 52, 57, 60, 64, 60, 57, 52]         # A2 E3 A3 C4 E4 C4 A3 E3
play(play_notes(presets["pluck"], arp))

换一个 preset,同一段音:

**应该听到:** 同样的音符,但每个音都是 "wow" 的音色(这里把 `mod_a` 调短了,不然 0.3 秒里来不及张开)。

In [ ]:
fast_wow = dict(presets["wow"], mod_a=0.08, mod_d=0.12)
play(play_notes(fast_wow, [33, 33, 40, 33, 36, 33, 43, 40]), gain=0.25)

## 8. 做自己的 preset

从一个现成的 preset 出发,改几个值。下面是一个起点,改数字、重新运行、看图、听。

想不出怎么改的话,试试这些目标:

- **更像笛子:** `wt_pos` 很低,`cut_env` 为 0,`amp_a` 0.08 左右
- **更像 "laser":** `cut_env` 5,`mod_d` 0.08,`res` 0.8
- **更像弦乐:** `wt_pos` 0.8,`amp_a` 0.3,`cutoff` 2000

In [ ]:
my_patch = dict(presets["wow"],
                note=21,
                res=0.85,
                cut_env=5.0)

show(my_patch)
play(synth(my_patch), gain=0.2)

**应该听到(没改之前):** 同一个 `wow`,低一个八度,resonance 更强,"wow" 更夸张。

## 常见错误

| 现象 | 原因 |
|---|---|
| `KeyError: 'cutof'` | 名字拼错了。`dict` 里没有这个名字 |
| `KeyError: 'mod_a'` | 把只有 6 个参数的 `test` 传给了需要 14 个参数的 `synth()` |
| 改了 preset 但声音没变 | 改完没有重新运行调用 `synth()` 的那个 cell |
| 声音很小或没有 | `cutoff` 很低而 `cut_env` 为 0;或者 `amp_s` 为 0 且 decay 很短 |
| 高音区有杂音 | `note` 太高,wavetable 里的 32 个 harmonic 超过了 sr / 2(第 3 集) |

## 练习

**1.** 做一个 `organ` preset:立刻出声、立刻停、音色不变。提示:`wt_env` 和 `cut_env` 都设 0。

<details><summary>答案</summary>

```python
organ = dict(note=57, wt_pos=0.4, wt_env=0.0, cutoff=3000, res=0.0, cut_env=0.0,
             amp_a=0.005, amp_d=0.01, amp_s=1.0, amp_r=0.01,
             mod_a=0.01, mod_d=0.01, mod_s=0.0, mod_r=0.01)
play(synth(organ))
```
</details>

**2.** 用循环听 `pad` 在 `res` 为 0、0.4、0.8 时的区别。

<details><summary>答案</summary>

```python
for value in [0, 0.4, 0.8]:
    display(play(synth(dict(presets["pad"], res=value)), gain=0.2))
```
</details>

**3.** 把琶音 `arp` 整体移高 5 个半音再弹。提示:循环里对每个音加 5。

<details><summary>答案</summary>

```python
higher = []
for m in arp:
    higher.append(m + 5)
play(play_notes(presets["pluck"], higher))
```
</details>

## 小结

| Max / RNBO | Python |
|---|---|
| 一个 RNBO patch | `synth(p)` 函数 |
| `param` object | `dict` 里的一项 |
| `dict` object | `dict` |
| preset | 一个 `dict` |
| `mtof` | `mtof(m)` |
| 拧旋钮 | `dict(旧的, 名字=新值)`,再调一次 `synth()` |

Python 里拧旋钮要 "重新算一遍",听起来比 Max 笨。但它换来一个 Max 很难做到的能力。

**下一集:** 让电脑自己拧旋钮,一口气生成一堆声音。